# BATTERY SOH PREDICTION USING NASA PCOE DATASET

# BUSINESS UNDERSTANDING

## Import Libraries

In [ ]:
import scipy.io
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
import shap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from scipy.ndimage import uniform_filter1d
from scipy import stats
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr 

## Data and Model Configuration

Intialize variables to store data and model configuration information

In [ ]:
DATA_DIR  = Path("./data")
RESULTS_DIR  = Path("./results")
RANDOM_STATE = 42

Initialize Variables to load fixed data objects

In [ ]:
BATTERIES = ["B0005", "B0006", "B0007", "B0018"] #B0036
q_rated = {
    "B0005": 2.00,
    "B0006": 2.00,
    "B0007": 2.00,
    "B0018": 2.00,
    "B0036": 2.00
}
df_qrated = pd.DataFrame.from_dict(q_rated, orient="index", columns=["Q_rated_Ah"])
df_qrated.index.name = "battery"
df_qrated = df_qrated.reset_index()
df_qrated

# DATA UNDERSTANDING

## Data Loading
Load one .mat file and return a flat DataFrame of all discharge timesteps.

Key findings from real files:
- Discharge cycles use Current_load and Voltage_load (not Current_charge/Voltage_charge)
- Capacity is a scalar float per cycle, broadcast across all timesteps
- Each cycle also carries ambient_temperature at the cycle level

In [ ]:
# Data Loading
def load_battery(battery_id: str):
    path = DATA_DIR / f"{battery_id}.mat"
    mat  = scipy.io.loadmat(str(path), simplify_cells=True)
    cycles = mat[battery_id]["cycle"]

    rows = []
    discharge_idx = 0

    for raw_cycle in cycles:
        if raw_cycle["type"] != "discharge":
            continue

        discharge_idx += 1
        d = raw_cycle["data"]
        n = len(d["Time"])
        capacity_Ah = float(d["Capacity"])

        for i in range(n):
            rows.append({
                "battery": battery_id,
                "cycle": discharge_idx,
                "ambient_temp_C": float(raw_cycle["ambient_temperature"]),
                "timestep": i,
                "Time": d["Time"][i],
                "Voltage_measured": d["Voltage_measured"][i],
                "Current_measured": d["Current_measured"][i],
                "Temperature_measured": d["Temperature_measured"][i],
                "Current_load": d["Current_load"][i],
                "Voltage_load": d["Voltage_load"][i],
                "capacity_Ah": capacity_Ah,
            })

    return pd.DataFrame(rows)

def load_all_cycles(battery_id):
    path = DATA_DIR / f"{battery_id}.mat"
    mat = scipy.io.loadmat(str(path), simplify_cells=True)
    cycles = mat[battery_id]["cycle"]

    charge_cycles = []
    discharge_caps = {}
    charge_idx = 0
    discharge_idx = 0

    for raw in cycles:
        if raw["type"] == "charge":
            charge_idx += 1
            charge_cycles.append({
                "battery": battery_id,
                "charge_cycle": charge_idx,
                "ambient_temp": float(raw["ambient_temperature"]),
                "data": raw["data"]
            })

        elif raw["type"] == "discharge":
            discharge_idx += 1
            cap = float(raw["data"]["Capacity"])
            discharge_caps[discharge_idx] = cap

    return charge_cycles, discharge_caps

In [ ]:
# CV phase current decay
def exponential_cv(t, HF4, HF5, HF6):
    return HF4 + HF5 * np.exp(-t / HF6)

In [ ]:
# Merge all battery data into a single DataFrame
raw_dfs = []
for bat in BATTERIES:
    df_bat = load_battery(bat)
    raw_dfs.append(df_bat)
    print(f"{bat}: {df_bat['cycle'].nunique():>3} discharge cycles, "
          f"{len(df_bat):>6,} timesteps")

raw_df = pd.concat(raw_dfs, ignore_index=True)

# Sanity checks
print(raw_df[["Voltage_measured","Current_measured",
              "Temperature_measured","Time","capacity_Ah"]].describe().round(4))

In [ ]:
raw_df.head()

### What is State of Health (SOH) in Battery technology?
Extract the cycle capacity (SOH): Target Variable for Regression Analysis

SOH is a percentage measure of how much usable capacity remains compared to when the battery was new.\
Calculation: SOH(t) = Q_measured(t) / Q_rated \
\
It directly coincides with estimating the Remaining Useful Life (RUL) with respect to defined threshold of capacity fade.

In [ ]:
# Cycle-level capacity table (target variable)
cycle_capacity = raw_df.groupby(["battery","cycle"])["capacity_Ah"].mean().reset_index()
cycle_capacity = pd.merge(cycle_capacity, df_qrated, how="left",on="battery")
cycle_capacity["SoH"] = np.round(cycle_capacity["capacity_Ah"] / cycle_capacity["Q_rated_Ah"],4) 
print(cycle_capacity[["capacity_Ah","SoH"]].sum())
cycle_capacity.head()

In [ ]:
print(cycle_capacity['battery'].unique())
cycle_capacity.describe()

### Electrochemical Impedance Spectroscopy (EIS)

In the NASA dataset the order per experiment block is:\
charge → discharge → impedance (repeating)\
We match impedance n to discharge n by co-increment

```
Re: electrolyte resistance (Ohm): high-frequency EIS intercept
Rct: charge transfer resistance (Ohm): semicircle diameter
```

In [ ]:
def load_eis_features(battery_id):
    path = DATA_DIR / f"{battery_id}.mat"
    mat = scipy.io.loadmat(str(path), simplify_cells=True)
    cycles = mat[battery_id]["cycle"]

    records = []
    discharge_idx = 0
    impedance_idx = 0

    for raw in cycles:
        ctype = raw["type"]
        if ctype == "discharge":
            discharge_idx += 1
        elif ctype == "impedance":
            impedance_idx += 1
            d = raw["data"]
            # print(d.keys())
            Re = float(np.array(d.get("Re", np.nan)).flat[0])
            Rct = float(np.array(d.get("Rct", np.nan)).flat[0])

            records.append({
                "battery": battery_id,
                "cycle":discharge_idx,
                "Re": Re,
                "Rct": Rct
            })

    return pd.DataFrame(records)


# Load EIS features for all batteries
eis_dfs = []
for bat in BATTERIES:
    df_eis = load_eis_features(bat)
    df_eis = df_eis.groupby(["battery", "cycle"]).agg({
        "Re": "mean",
        "Rct": "mean"
    }).reset_index()
    eis_dfs.append(df_eis)
    print(f"{bat}: {len(df_eis)} impedance cycles  "
          f"Re range: {df_eis['Re'].min():.4f}–{df_eis['Re'].max():.4f} ohm  "
          f"Rct range: {df_eis['Rct'].min():.4f}–{df_eis['Rct'].max():.4f}ohm")

eis_df = pd.concat(eis_dfs, ignore_index=True)

print(f"\nEIS table shape: {eis_df.shape}")
print(f"Nulls: {eis_df[['Re','Rct']].isnull().sum().to_dict()}")
print(f"Re mean±std: {eis_df['Re'].mean():.4f} ± {eis_df['Re'].std():.4f} ohm")
print(f"Rct mean±std: {eis_df['Rct'].mean():.4f} ± {eis_df['Rct'].std():.4f} ohm")
eis_df

## EXPLORATORY DATA ANALYSIS (EDA)

### EDA questions to contemplate:

1. Does capacity actually degrade?
2. Is the degradation monotonic?
3. Do the raw signals carry the aging signal and are they good indicators of aging?
4. Are the four batteries comparable? What is the extent of Inter-battery Variance for cross-battery testing? 

End of Life Threshold is set at 70% of the total capacity for each specific battery type.

In [ ]:
# Inputs:
#   raw_df: flat time-series, all batteries, discharge only
#   cycle_capacity: one row per cycle with capacity_Ah and SoH

# Consistent color per battery across all plots
BAT_COLORS = {
    "B0005": "#1f77b4",
    "B0006": "#ff7f0e",
    "B0007": "#2ca02c",
    "B0018": "#d62728"
}

# Define End of Life Threshold
EOL_THRESHOLD = 1.4   # Ah: 70%

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

### Plot 1: Capacity fade over cycles (the core degradation signal)
Plot the battery capacity across each discharge cycle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw capacity per cycle
ax = axes[0]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    ax.plot(sub["cycle"], sub["capacity_Ah"],
            color=color, alpha=0.55, linewidth=1.0, label=bat)
    # Overlay smoothed trend
    smoothed = uniform_filter1d(sub["capacity_Ah"].values, size=10)
    ax.plot(sub["cycle"], smoothed,
            color=color, linewidth=2.2)

ax.axhline(EOL_THRESHOLD, color="black", linestyle="--",
           linewidth=1.2, label=f"EOL threshold ({EOL_THRESHOLD} Ah)")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity (Ah)")
ax.set_title("Capacity fade over cycles\n(faint = raw, bold = 10-cycle smoothed)")
ax.legend(fontsize=9)

ax = axes[1]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    ax.plot(sub["cycle"], sub["SoH"],
            color=color, alpha=0.55, linewidth=1.0, label=bat)
    smoothed = uniform_filter1d(sub["SoH"].values, size=10)
    ax.plot(sub["cycle"], smoothed,
            color=color, linewidth=2.2)

ax.axhline(EOL_THRESHOLD / 2.0, color="black", linestyle="--",
           linewidth=1.2, label="EOL at SoH = 0.70")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("State of Health (SoH = capacity / Q-rated-capacity)")
ax.set_title("Normalised SoH over cycles")
ax.legend(fontsize=9)

plt.suptitle("Plot 1: Core degradation signal", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot1_capacity_fade.png", dpi=150, bbox_inches="tight")
plt.show()

print("Degradation summary")
print(
    cycle_capacity.groupby("battery").agg(
        n_cycles=("cycle", "count"),
        cap_start=("capacity_Ah", "max"),
        cap_end=("capacity_Ah", "min"),
        total_fade=("capacity_Ah", lambda x: x.max() - x.min()),
        fade_pct=("capacity_Ah", lambda x: 100*(x.max()-x.min())/x.max()),
    ).round(3)
)

#### Observation
We observe clear capacity fade from plot 1. Battery B0006 has the mode fade at 43% in 168 cycles. This attends to the core business issue of reduction in battery capacity and the need to plan the battery replacement adequately before the 70% degredation level is reached.

Further, the peaks indicate capacity degradation is non-monotonic. This implies the use of rolling statistics to capture the peaks as indications of further degradation. The peaks are known electrochemical phenomenon called capacity regeneration, caused by lithium redistribution during rest periods between test sessions. They are real signal, not measurement error.

From right pane, we observe degradation range of B0006 is deeper than others. Since B0018 is part of test data and others form the training data, the test distribution is within the range of training distribution. Hence, the inter-battery test/train split does not affect metrics in the current configuration. Battery agnostic features are imperative to capture true patterns.

### Plot 2: Discharge voltage curves across age (the raw signal)
Plot the intr-cycle voltage curves during battery discharge. 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, bat in enumerate(BATTERIES):
    ax = axes[idx]
    sub = raw_df[raw_df["battery"] == bat]
    n_cycles = sub["cycle"].nunique()

    # Sample 8 representative cycles spread across the battery's life
    sample_cycles = np.linspace(1, n_cycles, 8, dtype=int)
    cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

    for i, cyc in enumerate(sample_cycles):
        cyc_data = sub[sub["cycle"] == cyc].sort_values("Time")
        ax.plot(
            cyc_data["Time"] / 60,          # in minutes
            cyc_data["Voltage_measured"],
            color=cmap(i), linewidth=1.6,
            label=f"Cycle {cyc}"
        )

    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Voltage (V)")
    ax.set_title(f"{bat} - discharge curves\n(green=early, red=late)")
    ax.legend(fontsize=7.5, loc="lower left")
    ax.set_ylim(2.4, 4.4)

plt.suptitle("Plot 2: Voltage curve evolution across battery life", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot2_voltage_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Discharge duration summary (minutes, per battery)")
duration_df = (
    raw_df.groupby(["battery","cycle"])["Time"]
    .max()
    .reset_index()
    .rename(columns={"Time": "duration_s"})
)
duration_df["duration_min"] = duration_df["duration_s"] / 60
print(
    duration_df.groupby("battery")["duration_min"]
    .agg(["min","max","mean"])
    .round(1)
)

#### Observation
Red Curves (later cycles) have steeper initial drop (higher internal resistance), delayed plateau and reaches lowest voltage earlier. This indicates the Voltage curves clearly demarcates the aging of batteries. This proposes the usage of change in internal resistance, discharge duration, voltage slope and change in quantiles as features as these are distinct to earlier and later cycles.

### Plot 3: Temperature evolution across cycles
Plot the changes in temperature by different cycles of battery.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: temperature curves for one battery across age
ax = axes[0]
bat = "B0005"
sub = raw_df[raw_df["battery"] == bat]
n_cycles = sub["cycle"].nunique()
sample_cycles = np.linspace(1, n_cycles, 8, dtype=int)
cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

for i, cyc in enumerate(sample_cycles):
    cyc_data = sub[sub["cycle"] == cyc].sort_values("Time")
    ax.plot(cyc_data["Time"] / 60, cyc_data["Temperature_measured"],
            color=cmap(i), linewidth=1.6, label=f"Cycle {cyc}")

ax.set_xlabel("Time (min)")
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"{bat} - temperature curves across age\n(green=early, red=late)")
ax.legend(fontsize=7.5)

# Right: peak temperature per cycle for all batteries
ax = axes[1]
for bat, color in BAT_COLORS.items():
    sub = raw_df[raw_df["battery"] == bat]
    peak_temp = (
        sub.groupby("cycle")["Temperature_measured"]
        .max()
        .reset_index()
    )
    ax.scatter(peak_temp["cycle"], peak_temp["Temperature_measured"],
               color=color, s=8, alpha=0.6, label=bat)
    smoothed = uniform_filter1d(peak_temp["Temperature_measured"].values, size=10)
    ax.plot(peak_temp["cycle"], smoothed, color=color, linewidth=2.0)

ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Peak temperature (°C)")
ax.set_title("Peak temperature per cycle for all batteries\n"
             "(rising trend = increasing internal heat generation)")
ax.legend(fontsize=9)

plt.suptitle("Plot 3: Temperature as an aging indicator",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot3_temperature.png", dpi=150, bbox_inches="tight")
plt.show()

print("Peak temperature: first 10 vs last 10 cycles per battery")
for bat in BAT_COLORS:
    sub = raw_df[raw_df["battery"] == bat]
    peak = sub.groupby("cycle")["Temperature_measured"].max()
    print(f"{bat}:  early avg = {peak.iloc[:10].mean():.2f}°C  |  "
          f"late avg = {peak.iloc[-10:].mean():.2f}°C  |  "
          f"rise = {peak.iloc[-10:].mean() - peak.iloc[:10].mean():.2f}°C")

#### Observation
Clearly the initial and final temperatures vary according to cycles. This provides temperature as a physical indicative feature for training. The high inter-battery variance proposes to avoid max temperature as a feature. Relatively the rise in temperature will be a better feature for inter-battery model.

### Plot 4: Incremental capacity (dQ/dV) across age
Physics informed metric that demonstrates the rate of change of current with respect to voltage differences across cycles.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, bat in enumerate(BATTERIES):
    ax = axes[idx]
    sub = raw_df[raw_df["battery"] == bat]
    n_cycles = sub["cycle"].nunique()
    sample_cycles = np.linspace(1, n_cycles, 6, dtype=int)
    cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

    for i, cyc in enumerate(sample_cycles):
        cyc_data = sub[sub["cycle"] == cyc].sort_values("Voltage_measured")
        V = cyc_data["Voltage_measured"].values
        # Cumulative charge (Ah) via trapezoidal integration of |I| over time
        cyc_time = sub[sub["cycle"] == cyc].sort_values("Time")
        Q_cum = np.cumsum(
            np.abs(cyc_time["Current_measured"].values) *
            np.gradient(cyc_time["Time"].values)
        ) / 3600

        # Map Q onto the voltage-sorted axis
        sort_idx = np.argsort(cyc_time["Voltage_measured"].values)
        V_s = cyc_time["Voltage_measured"].values[sort_idx]
        Q_s = Q_cum[sort_idx]

        # Smooth before differentiating to reduce noise
        Q_smooth = uniform_filter1d(Q_s, size=8)
        with np.errstate(divide="ignore", invalid="ignore"):
            dV = np.gradient(V_s)
            dQdV = np.where(np.abs(dV) > 1e-4,
                            np.gradient(Q_smooth) / dV, 0)

        # Clip outliers for readable plot
        dQdV = np.clip(dQdV, 0, np.percentile(dQdV, 97))

        ax.plot(V_s, dQdV, color=cmap(i), linewidth=1.5,
                alpha=0.8, label=f"Cycle {cyc}")

    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("dQ/dV (Ah/V)")
    ax.set_title(f"{bat} - Incremental capacity (dQ/dV)\n(green=early, red=late)")
    ax.set_xlim(2.5, 4.3)
    ax.legend(fontsize=7.5)

plt.suptitle("Plot 4: dQ/dV: electrochemical fingerprint of degradation",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot4_dqdv.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observation
In early cycles the voltage changes smoothly and gradually across the full range. Hence, dQ/dV stays low and flat with no sharp peak forms. In late cycles the battery reaches the knee region faster, and the voltage plateau becomes more pronounced relative to the total discharge duration.

feature engineering:
Based on understanding from theoretical behavior the peaks are expected in earlier cycles, but the curves show a reversed trend. The peaks present are not truly visible as they are superposed by larger jumps due to scaling. It becomes important to normalise dQdV_peak by discharge duration and compute it only over the low-voltage regionwhere the physical peak actually lives, not over the full voltage range. Voltage position (where the peak occurs on the voltage axis) can be an additional feature.

### Plot 5: Correlation heatmap of candidate features vs target
To eliminate multi-collinearity among independent variables.

In [ ]:
def quick_features(df_bat):
    records = []
    for (bat, cyc), grp in df_bat.groupby(["battery","cycle"]):
        grp = grp.sort_values("Time")
        V = grp["Voltage_measured"].values
        I = grp["Current_measured"].values
        T = grp["Temperature_measured"].values
        t = grp["Time"].values
        dt = np.gradient(t)

        Q_cum = np.trapezoid(np.abs(I), t) / 3600
        E_Wh = np.trapezoid(np.abs(V * I), t) / 3600
        dV_init = np.diff(V[:5]).mean()
        dI_init = np.diff(I[:5]).mean()
        R_int = abs(dV_init / dI_init) if abs(dI_init) > 1e-4 else np.nan

        records.append({
            "battery": bat,
            "cycle": cyc,
            "discharge_dur_s": t[-1] - t[0],
            "voltage_mean": V.mean(),
            "voltage_std": V.std(),
            "voltage_slope": np.polyfit(t, V, 1)[0],
            "temp_max": T.max(),
            "temp_rise": T.max() - T[0],
            "R_int_proxy": R_int,
            "energy_Wh": E_Wh,
            "Q_cum_Ah": Q_cum,
            "capacity_Ah": grp["capacity_Ah"].iloc[0]
        })
    return pd.DataFrame(records)

feat_preview = quick_features(raw_df)

feat_cols = ["discharge_dur_s","voltage_mean","voltage_std","voltage_slope",
             "temp_max","temp_rise","R_int_proxy","energy_Wh","Q_cum_Ah","cycle"]

corr = feat_preview[feat_cols + ["capacity_Ah"]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True   # show lower triangle only

sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={"size": 9}
)
ax.set_title("Plot 5: Feature correlation matrix\n"
             "(bottom row = correlation with capacity_Ah target)",
             fontweight="bold")
plt.tight_layout()
plt.savefig("./assets/plot5_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Correlation with capacity_Ah")
print(
    corr["capacity_Ah"]
    .drop("capacity_Ah")
    .sort_values(key=abs, ascending=False)
    .round(3)
)

#### Observation
Multi-collinearity causes redundant features for linear modelling. Parameters across features become sensitive to each other it violates the assumption of independent variables in linear baseline. But as ensemble models are less affected all three features can be retained.

### Plot 6: Per-battery degradation comparison 
Validates the cross-battery split strategy for training and test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlay normalised to each battery's own starting capacity
ax = axes[0]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat].copy()
    sub["SoH_relative"] = sub["capacity_Ah"] / sub["capacity_Ah"].iloc[0]
    ax.plot(sub["cycle"], sub["SoH_relative"],
            color=color, linewidth=1.8, label=bat, alpha=0.85)

ax.axhline(0.80, color="black", linestyle="--", linewidth=1.0,
           label="80% of initial (common EOL proxy)")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity relative to cycle-1 capacity")
ax.set_title("Relative capacity fade\n(each battery normalised to its own start)")
ax.legend(fontsize=9)

# Right: cycle-by-cycle difference between batteries (B0005 as reference)
ax = axes[1]
ref = (cycle_capacity[cycle_capacity["battery"] == "B0005"]
       .set_index("cycle")["capacity_Ah"])

for bat in ["B0006", "B0007", "B0018"]:
    sub = (cycle_capacity[cycle_capacity["battery"] == bat]
           .set_index("cycle")["capacity_Ah"])
    common = ref.index.intersection(sub.index)
    diff = sub.loc[common] - ref.loc[common]
    ax.plot(common, diff, color=BAT_COLORS[bat],
            linewidth=1.5, label=f"{bat} − B0005")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity difference vs B0005 (Ah)")
ax.set_title("Inter-battery divergence\n(B0005 as reference)")
ax.legend(fontsize=9)

plt.suptitle("Plot 6: Cross-battery comparison (validates holdout split)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot6_battery_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Capacity loss per 10 cycles")
for bat in BAT_COLORS:
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    slope = np.polyfit(sub["cycle"], sub["capacity_Ah"], 1)[0]
    print(f"{bat}:  {slope*10:.4f} Ah / 10 cycles")

### EIS Plots: Visualizing electrochemical resistance and charge-transfer resistance

In [ ]:
BAT_COLORS = {
    "B0005": "#1f77b4", "B0006": "#ff7f0e",
    "B0007": "#2ca02c", "B0018": "#d62728"
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes,
    ["Re", "Rct"],
    ["Re (electrolyte resistance)", "Rct (charge transfer resistance)"]
):
    for bat, color in BAT_COLORS.items():
        sub = eis_df[eis_df["battery"] == bat].sort_values("cycle")
        ax.scatter(sub["cycle"], sub[col],
                   color=color, s=6, alpha=0.4, label=bat)
        if len(sub) > 10:
            smoothed = uniform_filter1d(sub[col].values, size=10)
            ax.plot(sub["cycle"], smoothed,
                    color=color, linewidth=2.0)
    ax.set_xlabel("Discharge cycle")
    ax.set_ylabel(f"{col} (Ω)")
    if col == "Re":
        ax.legend(fontsize=8)

plt.suptitle("Plot 25: EIS features Re and Rct vs cycle\n"
             "Direct electrochemical measurement of internal resistance components",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot25_eis_trends.png", dpi=150, bbox_inches="tight")
plt.show()


Add the EIS features Re and Rct to base data.

In [ ]:
# reference physical static data 
T_REF = 24.0   # reference ambient temperature (°C)
ALPHA = 0.005  # capacity temperature coefficient for LiCoO2 (Ah/°C)
VMIN_DQDV = 2.7 # voltage window for dQ/dV peak extraction
VMAX_DQDV = 3.7

def extract_cycle_features(grp):
    V = grp["Voltage_measured"].values
    I = grp["Current_measured"].values
    T = grp["Temperature_measured"].values
    t = grp["Time"].values
    T_amb = grp["ambient_temp_C"].iloc[0]

    # Statistical Features
    discharge_dur_s = t[-1] - t[0]
    voltage_mean = V.mean()
    voltage_std = V.std()
    voltage_slope = np.polyfit(t, V, 1)[0]   # V/s: encodes non-linear relations

    # Voltage at 80% of discharge duration: position-specific snapshot
    idx_80 = int(0.80 * len(t))
    voltage_at_80pct = V[idx_80]

    # temp_rise: [peak-start] to remove inter-battery ambient offset
    temp_rise = T.max() - T[0]

    # Physics-Informed features

    # Internal resistance at early stage: dV/dI at cycle start (first 10 steps)
    dV = np.diff(V[:10]).mean()
    dI = np.diff(I[:10]).mean()
    R_int_proxy = abs(dV / dI) if abs(dI) > 1e-4 else np.nan

    # Energy delivered this cycle (Wh)
    energy_Wh = np.trapezoid(np.abs(V * I), t) / 3600

    # Cumulative charge delivered (Ah)
    Q_cum_Ah = np.trapezoid(np.abs(I), t) / 3600

    # Temperature-compensated capacity
    #  Normalises Q_cum_Ah for ambient temperature drift across cycles
    #  Formula: Q_norm = Q / (1 + alpha * (T_amb - T_ref))
    Q_temp_compensated = Q_cum_Ah / (1 + ALPHA * (T_amb - T_REF))

    # dQ/dV peak features: computed over low-voltage window only
    mask = (V >= VMIN_DQDV) & (V <= VMAX_DQDV)
    dQdV_peak_height  = np.nan
    dQdV_peak_voltage = np.nan

    if mask.sum() > 10:
        V_win = V[mask]
        I_win = I[mask]
        t_win = t[mask]

        # Sort by voltage (ascending) for dQ/dV computation
        sort_idx = np.argsort(V_win)
        V_s = V_win[sort_idx]
        t_s = t_win[sort_idx]
        I_s = I_win[sort_idx]

        # Cumulative charge in this window
        Q_s = np.cumsum(np.abs(I_s) * np.gradient(t_s)) / 3600

        # Smooth Q before differentiating to reduce numerical noise
        Q_smooth = uniform_filter1d(Q_s, size=max(3, len(Q_s) // 15))

        dV_arr = np.gradient(V_s)
        with np.errstate(divide="ignore", invalid="ignore"):
            dQdV = np.where(np.abs(dV_arr) > 1e-5, np.gradient(Q_smooth) / dV_arr, 0)

        # Clip top 2% to suppress remaining noise spikes
        dQdV = np.clip(dQdV, 0, np.percentile(dQdV, 98))

        # Normalise by discharge duration
        dQdV_norm = dQdV / (discharge_dur_s / 3600)

        peak_idx = np.argmax(dQdV_norm)
        dQdV_peak_height  = dQdV_norm[peak_idx]
        dQdV_peak_voltage = V_s[peak_idx]

    return {
        # Statistical
        "discharge_dur_s": discharge_dur_s,
        "voltage_mean": voltage_mean,
        "voltage_std": voltage_std,
        "voltage_slope": voltage_slope,
        "voltage_at_80pct": voltage_at_80pct,
        "temp_rise": temp_rise,
        # Physics-informed
        "R_int_proxy": R_int_proxy,
        "energy_Wh": energy_Wh,
        "Q_cum_Ah": Q_cum_Ah,
        "Q_temp_compensated": Q_temp_compensated,
        "dQdV_peak_height": dQdV_peak_height,
        "dQdV_peak_voltage": dQdV_peak_voltage
    }

records = []
for (bat, cyc), grp in raw_df.groupby(["battery", "cycle"]):
    grp_sorted = grp.sort_values("Time")
    feat = extract_cycle_features(grp_sorted)
    feat["battery"] = bat
    feat["cycle"] = cyc
    feat["capacity_Ah"] = grp_sorted["capacity_Ah"].iloc[0]
    records.append(feat)

feature_df = pd.DataFrame(records)

In [ ]:
feature_eis_df = feature_df.merge(
    eis_df[["battery","cycle","Re","Rct"]],
    on = ["battery","cycle"],
    how = "inner"
)

print(f"\nFeature table with EIS: {feature_eis_df.shape}")
print(f"Rows lost in merge: {len(feature_df) - len(feature_eis_df)} "
      f"(cycles without matched impedance)")
print(f"\nRe correlation with capacity_Ah  : "
      f"{feature_eis_df['Re'].corr(feature_eis_df['capacity_Ah']):.4f}")
print(f"Rct correlation with capacity_Ah  : "
      f"{feature_eis_df['Rct'].corr(feature_eis_df['capacity_Ah']):.4f}")
print(f"nulls in Re: {feature_eis_df['Re'].isnull().sum()}  |  nulls in Rct: {feature_eis_df['Rct'].isnull().sum()}")

for col in ["Re", "Rct"]:
    valid = feature_eis_df[["capacity_Ah", col]].dropna()
    r, p  = spearmanr(valid["capacity_Ah"], valid[col])
    print(f"{col}: Spearman r={r:.4f}  p={p:.6f}  n={len(valid)}")

In [ ]:
# EIS vs capacity 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ["Re", "Rct"]):
    for bat, color in BAT_COLORS.items():
        sub = feature_eis_df[feature_eis_df["battery"] == bat]
        ax.scatter(sub[col], sub["capacity_Ah"],
                   color=color, s=8, alpha=0.5, label=bat)
    ax.set_xlabel(f"{col} (ohm)")
    ax.set_ylabel("Capacity (Ah)")
    ax.set_title(f"{col} vs capacity_Ah\n"
                 f"r = {feature_eis_df[col].corr(feature_eis_df['capacity_Ah']):.3f}",
                 fontweight="bold")
    if col == "Re":
        ax.legend(fontsize=8)

plt.suptitle("Plot 26: EIS features vs SoH: direct electrochemical evidence",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot26_eis_vs_capacity.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observation
This indicates large variance across batteries during early and late phases. Cycle level RMSE is a necessity. Also, battery type encoding features should be avoided such as cycle sequence.

### EDA summary:

1. Degradation is real, measurable and non-linear. Requires complex models that accounts for non-linear relations.
2. The raw signals do carry the aging signal. But requre feature engineering to avoid pitfalls.
3. B0018 is a legitimate holdout. Its degradation trajectory differs from the training batteries in rate and shape, making the cross-battery split a genuine generalisation test rather than an easy one.
4. Key features are highly correlated with the target. Ex: discharge duration, cumulative charge, and energy. Physics-informed features like internal resistance and temp rise provide additional signal shape info.
5. Re and Rct both show increasing trend with cycle number confirming electrochemical aging.